# TTS Playground : Try Out the TTS Models

In [ ]:
!pip install -U -q soundfile
!pip install -U -q "transformers>=4.52" accelerate torch torchaudio torchvision torchcodec
!pip install --no-cache-dir --no-dependencies "kittenml==0.9.8"
!pip install misaki num2words spacy onnxruntime espeakng-loader

## TTS Model 1 : KittenTTS

In [1]:
from kittenml import KittenTTS

kitten = KittenTTS("KittenML/kitten-tts-2-mini")
print(kitten.available_voices)

for v in kitten.voices("en"):
    print(v["name"], "-", v["description"])

for v in kitten.voices("de"):
    print(v["name"], "-", v["description"])

{'en': ['Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent', 'Max', 'Danny'], 'de': ['Lena', 'Greta', 'Jonas', 'Felix', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'es': ['Lucia', 'Carmen', 'Javier', 'Diego', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'it': ['Giulia', 'Chiara', 'Marco', 'Luca', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'pt': ['Beatriz', 'Camila', 'Rafael', 'Thiago', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'ru': ['Anastasia', 'Natasha', 'Dmitri', 'Ivan', 'Eleanor', 'Martha', 'Serena', 'Iris', 'Grace', 'Edith', 'Diana', 'Otis', 'Vincent'], 'hi': ['Priya', 'Ananya', 'Arjun', 'Vikram'], 'da': ['Freja', 'Mads', 'Rasmus'], 'fi': ['Aino', 'Helmi', 'Eero', 'Juho', 'Ville'], 'fr': ['Camille', 'Chloe', 'Manon', 'Louis', 'Antoine', 'Julien', 'Mathis', 'Etienne'], 'ja': ['Yuki'

In [2]:
import soundfile as sf
from IPython.display import Audio, display

text = "Hello there, this is Kitten TTS 2 Mini."   # change the text
voice = "Eleanor"   # the voice sets the language; e.g. "Lena" or "Felix" for German
speed = 1.0   # 1.1 for slightly faster speech

audio = kitten.generate(text, voice=voice, speed=speed)
sf.write("kitten_out.wav", audio, 24000)

display(Audio("kitten_out.wav", rate=24000))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/50 [00:00<?, ?it/s]

/opt/conda/lib/python3.14/site-packages/torch/jit/_script.py:1485: FutureWarning: `torch.jit.script` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [3]:
# generate by language (uses the language's first voice)
text = "Guten Morgen! Wie geht es dir?"   #change the text
language = "de"

audio = kitten.generate(text, language=language)
sf.write("kitten_out.wav", audio, 24000)

display(Audio("kitten_out.wav", rate=24000))

ModuleNotFoundError: No module named 'espeakng_loader'

## TTS Model 2 : Audio8 (Voice Cloning)

In [3]:
import torch
import soundfile as sf
from transformers import AutoModel, AutoProcessor

model_id = "Edge0/Audio8-TTS-Realistic-Air"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=dtype).eval().to(device)
print(f"Model loaded on {model.device} with dtype {model.dtype}")

Loading weights:   0%|          | 0/226 [00:00<?, ?it/s]

Model loaded on cuda:0 with dtype torch.bfloat16


In [4]:
#TODO: TEST THE VOICES FROM REFERENCE_VOICES.TXT
REFERENCE_WAV = "/training-1/tts/vctk/wav48_silence_trimmed/p225/p225_001_mic1.wav"   # a short reference recording
REFERENCE_TEXT = "Please call Stella."   # its transcript

from IPython.display import Audio, display

# listen to the reference first: the clone should sound like this speaker
ref_waveform, ref_sr = sf.read(REFERENCE_WAV)
display(Audio(ref_waveform, rate=ref_sr))

In [5]:
text = "Welcome to Audio8 TTS. It is wonderful to hear from you."   # change the text

inputs = processor(
    text=[text],
    reference_audio=[REFERENCE_WAV],
    reference_text=[REFERENCE_TEXT],
    return_tensors="pt",
)
inputs = {name: value.to(device) for name, value in inputs.items()}

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=0.8,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        return_dict_in_generate=True,
    )
    waveforms, waveform_lengths = model.decode_audio(output.codes)

audio = waveforms[0, : int(waveform_lengths[0])].float().cpu().numpy()
sf.write("audio8_out.wav", audio, model.config.codec_sample_rate)

display(Audio("audio8_out.wav", rate=model.config.codec_sample_rate))

/opt/conda/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
